# Milestone 4: Feature Engineering & Leak-Proof Splitting
**Renewable Energy-Aware Load Prediction Model for Smart Grids**  
*Pavagada Solar Park (2,050 MW) & Karnataka State Grid Baseline*

This notebook implements and validates **Milestone 4** of our engineering roadmap:
1. **Horizon-Aware Feature Engineering**: Constructing calendar, solar geometry, weather forecast, and target lag features strictly respecting forecast horizons ($h=1$ for $S_1, D_1$ and $h=24$ for $S_{24}, D_{24}$).
2. **Strict Chronological Splitting**: Partitioning data into Train (2021-03 to 2024-12), Validation (2025), and Quarantined Test (2026-01 to 2026-02) with zero temporal shuffling.
3. **Empirical Anti-Leakage Perturbation Test**: Proving that perturbing future target/weather data has zero impact on features extracted at prior times.
4. **Train-Only Scaler Fitting**: Fitting `StandardScaler` strictly on training data to prevent lookahead contamination.
5. **3D Sequence Window Construction**: Formatting sliding window tensors `(samples, 24, features)` for Tier 1-3 Deep Learning architectures (LSTM, GRU, CNN-LSTM).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Set up project root path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.features import (
    create_problem_features,
    split_chronological,
    fit_and_apply_scalers,
    build_sequences,
    process_and_save_all_problems
)

df_clean = pd.read_parquet(config.CLEAN_DATA_PARQUET)
print(f'Clean hourly dataset loaded: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
print(f'Spanning: {df_clean["timestamp_ist"].min()} to {df_clean["timestamp_ist"].max()}')

Clean hourly dataset loaded: 43,824 rows x 47 columns
Spanning: 2021-03-01 00:00:00 to 2026-02-28 23:00:00


## 1. Feature Space Architecture and Horizon Constraints

For each of the four forecasting problems, features are engineered with strict respect for the physical and operational forecast horizon $h$:
- **Common Base Features (32 channels)**:
  - **10 Calendar & Cyclical Signals**: `hour_sin`, `hour_cos`, `doy_sin`, `doy_cos`, `month_sin`, `month_cos`, `dow_sin`, `dow_cos`, `is_weekend`, `is_holiday`.
  - **22 NWP Weather & Astronomical Forecasts at Target Hour $t$**: GHI, DNI, DHI, POA, clearsky GHI, clearsky index, temperature, module temperature, relative humidity, wind speed, wind direction, total/low/mid/high cloud covers, precipitation, surface pressure, aerosol optical depth, solar zenith, azimuth, elevation, and daylight indicator.
- **Problem-Specific Autoregressive Lags & Rolling Statistics**:
  - **$S_1$ / $D_1$ ($h=1$, 1-hour ahead)**: Allowed lags are $t-1, t-2, t-3, t-24, t-48, t-168$. Rolling statistics (3h and 24h mean/std) are computed on the target shifted by 1 hour.
  - **$S_{24}$ / $D_{24}$ ($h=24$, 24-hours ahead / Day-ahead dispatch)**: Allowed lags are $t-24, t-48, t-168$ only. Lags $< 24$ are strictly forbidden. Rolling statistics (24h and 168h mean/std) are shifted by 24 hours.

In [2]:
# Inspect feature dimensions for all four problems
problem_summary = []
for pid in ['S1', 'S24', 'D1', 'D24']:
    X, y, features = create_problem_features(df_clean, pid)
    problem_summary.append({
        'Problem ID': pid,
        'Description': config.PROBLEMS[pid]['description'],
        'Target': config.PROBLEMS[pid]['target'],
        'Horizon (h)': f"{config.PROBLEMS[pid]['horizon_hours']} hr",
        'Total Features': len(features),
        'Total Samples': len(X),
        'Earliest Sample': str(X.index.min()),
        'Latest Sample': str(X.index.max())
    })

summary_df = pd.DataFrame(problem_summary)
display(summary_df)

,Problem ID,Description,Target,Horizon (h),Total Features,Total Samples,Earliest Sample,Latest Sample
0,S1,Solar power 1-hour ahead,solar_power_mw,1 hr,44,43656,2021-03-08 00:00:00,2026-02-28 23:00:00
1,S24,Solar power 24-hours ahead,solar_power_mw,24 hr,41,43656,2021-03-08 00:00:00,2026-02-28 23:00:00
2,D1,Demand 1-hour ahead,demand_mw,1 hr,44,43656,2021-03-08 00:00:00,2026-02-28 23:00:00
3,D24,Demand 24-hours ahead,demand_mw,24 hr,41,43656,2021-03-08 00:00:00,2026-02-28 23:00:00


## 2. Chronological Splitting (Train, Validation, Test)

In accordance with time-series machine learning standards, **temporal order is strictly preserved** and data is never randomly shuffled:
- **Train Split (33,480 samples)**: 2021-03-08 00:00 to 2024-12-31 23:00 (approx. 76.7% of total valid observations). The first 168 hours of the raw dataset are absorbed as initial lag burn-in.
- **Validation Split (8,760 samples)**: 2025-01-01 00:00 to 2025-12-31 23:00 (exactly one full 365-day annual cycle, 20.1% of data). Used for model validation, tuning, and tier progression.
- **Test Split (1,416 samples)**: 2026-01-01 00:00 to 2026-02-28 23:00 (3.2% of data). **Quarantined until Milestone 10**.

In [3]:
# Verify split boundaries and sample counts for S1
X_s1, y_s1, _ = create_problem_features(df_clean, 'S1')
splits_s1 = split_chronological(X_s1, y_s1)

total_samples = len(X_s1)
split_table = pd.DataFrame([
    {
        'Split': 'Train',
        'Start': splits_s1['X_train'].index.min(),
        'End': splits_s1['X_train'].index.max(),
        'Samples': len(splits_s1['X_train']),
        'Share (%)': f"{len(splits_s1['X_train']) / total_samples * 100:.2f}%"
    },
    {
        'Split': 'Validation',
        'Start': splits_s1['X_val'].index.min(),
        'End': splits_s1['X_val'].index.max(),
        'Samples': len(splits_s1['X_val']),
        'Share (%)': f"{len(splits_s1['X_val']) / total_samples * 100:.2f}%"
    },
    {
        'Split': 'Test (Quarantined)',
        'Start': splits_s1['X_test'].index.min(),
        'End': splits_s1['X_test'].index.max(),
        'Samples': len(splits_s1['X_test']),
        'Share (%)': f"{len(splits_s1['X_test']) / total_samples * 100:.2f}%"
    }
])
display(split_table)

# Confirm zero boundary overlap
assert splits_s1['X_train'].index.max() < splits_s1['X_val'].index.min(), 'Train/Val overlap error'
assert splits_s1['X_val'].index.max() < splits_s1['X_test'].index.min(), 'Val/Test overlap error'
print('Chronological split assertions: PASSED (Zero boundary overlap)')

,Split,Start,End,Samples,Share (%)
0,Train,2021-03-08,2024-12-31 23:00:00,33480,76.69%
1,Validation,2025-01-01,2025-12-31 23:00:00,8760,20.07%
2,Test (Quarantined),2026-01-01,2026-02-28 23:00:00,1416,3.24%


Chronological split assertions: PASSED (Zero boundary overlap)


## 3. Empirical Anti-Leakage Verification (Future Perturbation Test)

A critical vulnerability in time series pipelines is subtle lookahead leakage (e.g., rolling statistics computed without an explicit backward shift, or backward-looking features inadvertently using future timestamps).  
To rigorously prove zero leakage, we execute an automated perturbation test:
1. We extract feature vector $x_t$ at pivot timestamp $t = \text{2023-06-15 12:00:00}$.
2. We corrupt the future dataset ($t' > t$) by setting future solar power to 999,999 MW, demand to 888,888 MW, and GHI to 777,777 $W/m^2$.
3. We re-run the entire feature extraction pipeline and compare $x_t^{\text{corrupted}}$ against $x_t$.
4. If there is zero lookahead leakage, $\max(|x_t - x_t^{\text{corrupted}}|) \equiv 0.0$.

In [4]:
pivot_ts = pd.Timestamp('2023-06-15 12:00:00')

corrupted_df = df_clean.copy()
corrupted_df['timestamp_ist'] = pd.to_datetime(corrupted_df['timestamp_ist'])
future_mask = corrupted_df['timestamp_ist'] > pivot_ts
corrupted_df.loc[future_mask, 'solar_power_mw'] = 999999.0
corrupted_df.loc[future_mask, 'demand_mw'] = 888888.0
corrupted_df.loc[future_mask, 'ghi_wm2'] = 777777.0
corrupted_df.loc[future_mask, 'temperature_2m_c'] = 666666.0

leakage_results = []
for pid in ['S1', 'S24', 'D1', 'D24']:
    X_orig, _, _ = create_problem_features(df_clean, pid)
    X_corr, _, _ = create_problem_features(corrupted_df, pid)
    
    row_orig = X_orig.loc[pivot_ts].values
    row_corr = X_corr.loc[pivot_ts].values
    max_abs_diff = float(np.max(np.abs(row_orig - row_corr)))
    leakage_results.append({
        'Problem': pid,
        'Pivot Timestamp': str(pivot_ts),
        'Max Feature Difference': f'{max_abs_diff:.10f}',
        'Leakage Status': 'PASSED (Zero Leakage)' if max_abs_diff < 1e-7 else 'FAILED'
    })

leakage_df = pd.DataFrame(leakage_results)
display(leakage_df)

,Problem,Pivot Timestamp,Max Feature Difference,Leakage Status
0,S1,2023-06-15 12:00:00,0.0000000000,PASSED (Zero Leakage)
1,S24,2023-06-15 12:00:00,0.0000000000,PASSED (Zero Leakage)
2,D1,2023-06-15 12:00:00,0.0000000000,PASSED (Zero Leakage)
3,D24,2023-06-15 12:00:00,0.0000000000,PASSED (Zero Leakage)


## 4. Train-Only Scaler Fitting

For models that require feature normalization (Linear models, Ridge/Lasso, SVR, MLP, and Deep Learning sequence models), fitting scalers on the entire dataset would leak future distribution parameters (mean and variance) into training.  
We fit `StandardScaler` strictly on `X_train` and transform `X_train`, `X_val`, and `X_test`. Saved to `results/models/scaler_{problem}.joblib`.

In [5]:
# Verify scaler fit on Train only
scaler_s1 = joblib.load(config.RESULTS_MODELS_DIR / 'scaler_S1.joblib')
X_train_s1 = pd.read_parquet(config.PROCESSED_DATA_DIR / 'X_train_S1.parquet')
X_val_s1 = pd.read_parquet(config.PROCESSED_DATA_DIR / 'X_val_S1.parquet')

print(f'Scaler mean matches Train mean: {np.allclose(scaler_s1.mean_, X_train_s1.mean().values, atol=1e-5)}')
print(f'Scaler mean matches Val mean:   {np.allclose(scaler_s1.mean_, X_val_s1.mean().values, atol=1e-2)}')

# Sample scaled vs unscaled feature values
comparison = pd.DataFrame({
    'Feature': X_train_s1.columns[:5],
    'Raw Mean': X_train_s1.mean().values[:5].round(3),
    'Raw Std': X_train_s1.std().values[:5].round(3),
    'Scaler Fitted Mean': scaler_s1.mean_[:5].round(3),
    'Scaler Fitted Scale': scaler_s1.scale_[:5].round(3)
})
display(comparison)

Scaler mean matches Train mean: True
Scaler mean matches Val mean:   False


,Feature,Raw Mean,Raw Std,Scaler Fitted Mean,Scaler Fitted Scale
0,hour_sin,0.000,0.707,0.000,0.707
1,hour_cos,0.000,0.707,0.000,0.707
2,doy_sin,-0.024,0.712,-0.024,0.712
3,doy_cos,-0.038,0.701,-0.038,0.701
4,month_sin,-0.039,0.703,-0.039,0.703


## 5. 3D Sequence Windows for Deep Learning Models

Deep learning sequence models (LSTM, GRU, CNN-LSTM, Attention, Transformers) require 3D input tensors:  
$$\text{Shape: } (N_{\text{samples}}, \text{window\_length}, N_{\text{features}})$$
where each sample consists of the preceding 24 hours of observations predicting the target at step $t$.

In [6]:
X_train_scaled = pd.read_parquet(config.PROCESSED_DATA_DIR / 'X_train_S1_scaled.parquet')
y_train = pd.read_parquet(config.PROCESSED_DATA_DIR / 'y_train_S1.parquet')['solar_power_mw']

window_size = 24
X_seq, y_seq = build_sequences(X_train_scaled.values, y_train.values, window_size=window_size)

print(f'Original 2D Tabular Training Shape: {X_train_scaled.shape}')
print(f'Constructed 3D Sequence Tensor:     {X_seq.shape}')
print(f'Corresponding Sequence Target:      {y_seq.shape}')
print(f'Samples Lost to Sequence Window:    {len(X_train_scaled) - len(X_seq)} samples ({window_size - 1} hours)')
assert X_seq.shape == (len(X_train_scaled) - window_size + 1, window_size, X_train_scaled.shape[1])
assert len(y_seq) == len(X_seq)
print('Sequence builder integrity check:  PASSED')

Original 2D Tabular Training Shape: (33480, 44)
Constructed 3D Sequence Tensor:     (33457, 24, 44)
Corresponding Sequence Target:      (33457,)
Samples Lost to Sequence Window:    23 samples (23 hours)
Sequence builder integrity check:  PASSED


## 6. Milestone 4 Checkpoint Conclusion

All deliverables for **Milestone 4: Features and Splits** are finalized and verified:
1. **Tabular Feature Sets**: Cleanly partitioned into Train, Validation, and Quarantined Test splits across all 4 problems ($S_1, S_{24}, D_1, D_{24}$).
2. **Zero Lookahead Leakage**: Verified through mathematical shifting and empirical future-perturbation tests.
3. **Physical Units & Scaled Tensors**: Raw parquets saved for gradient boosting algorithms (XGBoost/LightGBM/CatBoost) and scaled parquets/3D sequence builders ready for neural architectures.
4. **Artifacts Exported**: All datasets stored in `data/processed/`, scalers in `results/models/`, and metadata cataloged in `data/processed/feature_metadata.json`.